# Architecture Translation: Earn Every Component

> **The story.** In 1972, David Parnas argued that systems should be decomposed around decisions likely to change, not around a convenient sequence of processing steps. That discipline matters even more when a customer says `AI`, `agent`, or `use everything`: the label is not the architecture. Riverside House needs each component to earn its place against a workflow fact, a policy boundary, and a failure it can actually repair.
>
> **Where you are.** You receive frozen Riverside case version `RIV-FDE-1.0.0`: four bounded use cases, seven current workflow steps, synthetic baselines, policy constraints, 11 intentional conflicts, and 10 open unknowns. Discovery has framed the problem; it has not approved a solution. This notebook delivers `ARC-01`, `ARC-02`, `ADR-001`, and `ARC-03` for later stages to challenge.
>
> **Notation.** $O$ is an architecture option; $R$ is a Riverside requirement; $C(O)$ is the capability set supplied by option $O$; $B(O)$ is its hard-blocker set; $A$ is the exact action payload a human reviews.

> **Inter-notebook contract:** this notebook reads frozen shared fixtures directly. It writes no artifact, changes no case fact, calls no model, and contacts no service. Its tables are proposed teaching outputs until reviewed by authorized Riverside owners.

## 0 - The Challenge

> **The mission:** reduce Riverside's 18-minute median policy search and 42-minute median bounded-continuation draft time without weakening title access, regional processing, rights authority, or human control.

**What we know so far:**

- Policy search is 18 minutes median and 34 minutes p90 across 46 sampled tasks (`MET-RIV-001`, `MET-RIV-002`).
- Bounded continuation drafting is 42 minutes median across 31 sampled tasks (`MET-RIV-003`).
- The current-guidance first-result rate is 61% across 36 replayed policy queries (`MET-RIV-005`).
- Riverside has four explicit use cases and prohibits autonomous publication, rights, contract, royalty, payment, and supplier changes.
- Missing tenant, actor, role, region, purpose, title assignment, or trace context must fail closed (`SEC-RIV-002`).
- **But the brief already implies broad AI access and workflow automation before the smallest valid intervention has been established.**

**What's blocking us:** a plausible demo can hide the wrong source, an unauthorized title, a stale policy, an unapproved region, or a duplicate PageTurn write. Calling the product an agent resolves none of those failures.

**What this chapter unlocks:** an evidence-labeled option matrix, bounded architecture, ADR, and customer explanation that select the smallest composition and preserve explicit revisit triggers.

```mermaid
flowchart LR
    A["Customer says: instant, use everything, update workflow"] --> B["Failure: solution label hides requirements"]
    B --> C["Try no AI and deterministic controls first"]
    C --> D["Add only what a named failure requires"]
    D --> E["Stop when bounded use cases are covered"]
    E --> F["Reject unearned agentic complexity"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Topic-space contract

| Option | Coverage | Why |
|---|---|---|
| No AI / process repair | Built and checked | The anti-AI reviewer needs a real option |
| Deterministic software | Built and checked | Owns validation, authorization, lifecycle, and transitions |
| Search | Built and checked | Tests whether finding current authorized passages is enough |
| RAG | Built and checked | Adds cited synthesis but not authority |
| Prompt-only call | Built and checked | Tests bounded drafting with supplied context |
| Fine-tuning | Built and checked | Separates stable behavior from current facts and access |
| Deterministic workflow | Built and checked | Encodes known branches, approvals, retries, and recovery |
| Single agent | Built and checked | Requires evidence of runtime branch uncertainty |
| Multi-agent system | Built and checked | Requires independently complex dynamic work and coordination benefit |

In [ ]:
# ── Load Frozen Riverside Fixtures ───────────────────────────────────────
from pathlib import Path
import hashlib
import json
from pprint import pprint

CASE_CANDIDATES = [
    Path("../shared/fixtures/riverside-engagement-v1.json"),
    Path("learning/fde/shared/fixtures/riverside-engagement-v1.json"),
]
FACT_CANDIDATES = [
    Path("../shared/fixtures/expected-facts-v1.json"),
    Path("learning/fde/shared/fixtures/expected-facts-v1.json"),
]
CASE_PATH = next((path for path in CASE_CANDIDATES if path.exists()), None)
FACT_PATH = next((path for path in FACT_CANDIDATES if path.exists()), None)
if CASE_PATH is None or FACT_PATH is None:
    raise FileNotFoundError("Run from the chapter directory or repository root.")

case = json.loads(CASE_PATH.read_text(encoding="utf-8"))
fact_ledger = json.loads(FACT_PATH.read_text(encoding="utf-8"))
assert case["fixture_version"] == "RIV-FDE-1.0.0"
assert len(case["use_cases"]) == 4
assert len(case["intentional_conflicts"]) == 11
assert len(case["unknowns"]) == 10
assert {fact["fact_id"] for fact in fact_ledger["facts"]} >= {"FACT-RIV-002", "FACT-RIV-040"}
print("PASS: frozen Riverside fixture and expected-facts ledger loaded.")
print(f"Use cases: {len(case['use_cases'])}; conflicts: {len(case['intentional_conflicts'])}; unknowns: {len(case['unknowns'])}")
print("Takeaway: every architecture claim must stay traceable to these immutable inputs.")

## 1 - Start Where AI Is Absent

The first option is not a smaller model. It is no model. Riverside can remove superseded policies from current results, assign source owners, normalize title IDs, preserve manual review, and define exact transitions before inference exists.

No AI still fails two explicit outcomes: drafting bounded manuscript text and turning paraphrased questions plus several passages into a concise cited answer. That failure earns model-assisted paths later. It does not erase the process controls around them.

```mermaid
flowchart LR
    N["No AI: lifecycle, owners, manual fallback"] --> F1["Failure: no bounded draft"]
    N --> F2["Failure: weak paraphrase synthesis"]
    F1 --> P["Bounded prompt candidate"]
    F2 --> S["Authorized search candidate"]
    S --> F3["Failure: passages, not a supported answer"]
    F3 --> R["RAG candidate"]
    style N fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F1 fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F2 fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F3 fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** If Phase 0 excludes superseded policy, preserves manual review, and assigns owners, does it satisfy A) all four use cases, B) none, or C) the deterministic controls while still missing bounded drafting and cited synthesis? The next cell resolves the named capabilities.

### Common Pitfalls

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Treat `no AI` as doing nothing | Known lifecycle, identifier, approval, and ownership failures remain |
| Right | Make process and deterministic controls Phase 0 | They remain independently useful around later model calls |
| Wrong | Demand one option cover every use case | Search, generation, and workflow authority solve different problems |
| Right | Compose the smallest option per bounded use case | Each component has one reason to exist and one owner |

**Quick Health Check:** the anti-AI option must name a concrete improvement, a manual fallback, and the exact outcome it cannot meet.

**Reflection:** Phase 0 earns the control shell but fails two language outcomes. That residual failure justifies comparing search, RAG, and bounded prompting without granting any of them authority.

In [ ]:
# ── Prove the Anti-AI Option and Define Capability Surfaces ──────────────
USE_CASE_REQUIREMENTS = {
    "UC-RIV-001": {"authorized_current_sources", "cited_answer", "human_use_decision"},
    "UC-RIV-002": {"selected_title_context", "bounded_generation", "human_acceptance"},
    "UC-RIV-003": {"transition_proposal", "exact_payload_approval", "idempotent_commit"},
    "UC-RIV-004": {"authorized_rights_lookup", "human_legal_decision"},
}
OPTIONS = {
    "no_ai": {"manual_process", "source_lifecycle", "human_use_decision", "human_acceptance", "human_legal_decision"},
    "deterministic_software": {"source_lifecycle", "fail_closed_context", "transition_proposal", "exact_payload_approval"},
    "search": {"authorized_current_sources", "authorized_rights_lookup", "stable_source_ids"},
    "rag": {"authorized_current_sources", "cited_answer", "grounded_answer"},
    "prompt_call": {"selected_title_context", "bounded_generation"},
    "fine_tuning": {"stable_style", "bounded_generation"},
    "deterministic_workflow": {"transition_proposal", "exact_payload_approval", "idempotent_commit", "durable_state"},
    "single_agent": {"runtime_branch_selection", "model_selected_tools"},
    "multi_agent": {"runtime_branch_selection", "distributed_reasoning", "model_selected_tools"},
}
ANTI_AI_REVIEW = {
    "improvements": ["exclude_superseded_policy", "assign_source_owners", "preserve_manual_review"],
    "fallback": "manual workflow plus authorized current-source search",
    "unmet_outcomes": ["bounded_generation", "cited_synthesis_from_multiple_passages"],
}
assert ANTI_AI_REVIEW["improvements"]
assert "manual" in ANTI_AI_REVIEW["fallback"]
assert set(ANTI_AI_REVIEW["unmet_outcomes"]) == {"bounded_generation", "cited_synthesis_from_multiple_passages"}
for option_name in ("no_ai", "deterministic_software", "search", "rag", "prompt_call"):
    missing = {use_case_id: sorted(required - OPTIONS[option_name]) for use_case_id, required in USE_CASE_REQUIREMENTS.items() if required - OPTIONS[option_name]}
    print(f"{option_name}: missing capabilities -> {missing}")
print("PASS: no AI earns Phase 0 controls and a fallback, but misses two explicit language outcomes.")
print("Takeaway: no single label covers the engagement; Riverside needs a bounded composition.")

## 2 - Search, RAG, Prompting, and Fine-Tuning Solve Different Failures

Search returns passages. RAG supplies retrieved passages to a generator and returns an answer with evidence handles. A prompt-only call can draft when the selected manuscript context is already known. Fine-tuning can practice stable behavior, but model weights are not a current policy store, title ACL, citation, deletion mechanism, or rights decision.

The design separates two paths: `UC-RIV-001` needs current authorized retrieval plus cited synthesis; `UC-RIV-002` needs a bounded prompt against one explicitly selected title. Fine-tuning re-enters only if held-out evaluation isolates a stable style, format, or instruction gap.

```mermaid
flowchart LR
    Q["Policy question"] --> S["Authorized search"]
    S --> SF["Failure: passages require synthesis"]
    SF --> R["RAG with citations"]
    M["Selected manuscript + editor instruction"] --> P["Bounded prompt call"]
    P --> PF["Possible stable behavior gap"]
    PF --> T["Fine-tuning after repeated evidence"]
    R --> H["Human decides whether to use answer"]
    T --> H2["Editor accepts or rejects draft"]
    style Q fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style SF fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style M fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style PF fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style T fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H2 fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Common Pitfalls

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Fine-tune policies and rights into weights | Current versions, citations, ACLs, and deletion become unprovable |
| Right | Retrieve current authorized facts; tune only stable behavior | Knowledge and behavior keep separate release paths |
| Wrong | Call any generated answer `RAG` | RAG requires traceable retrieval, not just a long prompt |
| Right | Evaluate retrieval, citation support, and answer quality separately | Fluent text from the wrong source is still wrong |
| Wrong | Let the model choose region or authorization filters | Model text is not a policy decision |
| Right | Resolve region and ACLs before retrieval or inference | Missing or conflicting context fails closed |

**Quick Health Check:** every selected model path names its approved input, evidence source, human decision, and AI-off fallback. Fine-tuning remains unable to satisfy authorization by construction.

In [ ]:
# ── Build ARC-01 and Check Authority Separation ──────────────────────────
ARC_01 = [
    {"option": "No AI / process repair", "fit": "required Phase 0", "failure": "cannot draft or synthesize paraphrased evidence", "disposition": "select", "revisit": "never remove manual fallback"},
    {"option": "Deterministic software", "fit": "identity, lifecycle, schemas, transition rules", "failure": "cannot interpret open-ended language", "disposition": "select as control shell", "revisit": "not applicable"},
    {"option": "Search", "fit": "authorized current passages and stable source IDs", "failure": "returns passages rather than a composed answer", "disposition": "select and preserve as degraded mode", "revisit": "measure whether synthesis adds value"},
    {"option": "RAG", "fit": "UC-RIV-001 cited policy answers", "failure": "generation is not authority and needs regional approval", "disposition": "select read-only path", "revisit": "remove if search-only meets acceptance"},
    {"option": "Prompt-only model call", "fit": "UC-RIV-002 bounded continuation", "failure": "does not discover policy or enforce source ACLs", "disposition": "select explicit-title path", "revisit": "remove if no workflow benefit"},
    {"option": "Fine-tuning", "fit": "stable style, format, or instruction behavior", "failure": "weights cannot own facts, citations, ACLs, or deletion", "disposition": "defer", "revisit": "stable behavior gap measured"},
    {"option": "Deterministic workflow", "fit": "UC-RIV-003 known states, approval, retry, recovery", "failure": "PageTurn idempotency unresolved", "disposition": "select design; disable writes", "revisit": "UNK-RIV-005 closed and failure injection measured"},
    {"option": "Single agent", "fit": "runtime selection among unanticipated actions", "failure": "no frozen use case requires model-selected control flow", "disposition": "reject", "revisit": "valuable unenumerable branch measured"},
    {"option": "Multi-agent system", "fit": "independently complex dynamic reasoning", "failure": "no decomposition need; trust and convergence risks multiply", "disposition": "reject", "revisit": "single-agent bottleneck and coordination gain measured"},
]
AUTHORITY_CAPABILITIES = {"fail_closed_context", "exact_payload_approval", "human_legal_decision"}
MODEL_OPTIONS = {"rag", "prompt_call", "fine_tuning", "single_agent", "multi_agent"}
for option_name in MODEL_OPTIONS:
    assert not (OPTIONS[option_name] & AUTHORITY_CAPABILITIES)
assert {"authorized_current_sources", "cited_answer", "fail_closed_context"} - OPTIONS["fine_tuning"] == {"authorized_current_sources", "cited_answer", "fail_closed_context"}
assert len(ARC_01) == 9
for row in ARC_01:
    print(f"{row['option']:<28} | {row['disposition']:<32} | failure: {row['failure']}")
print("PASS: all nine options have Riverside-specific failures and model-backed options own no authority.")

## 3 - Workflow Before Agent, One Agent Before Many

A workflow executes a graph authored in advance. An agent lets a model choose the next action at runtime. That freedom is valuable only when representative tasks contain important branches that cannot be enumerated economically. Riverside's four use cases already name the actor, allowed action, prohibited action, and human decision. That is workflow-shaped evidence.

Multiple business teams do not imply multiple agents. Multi-agent design needs independently complex dynamic reasoning, bounded delegation, typed messages, shrinking authority, convergence, and evidence that coordination benefit exceeds added latency, cost, evaluation, and incident surface.

```mermaid
flowchart TD
    A{"Can valid next actions be enumerated?"} -->|"Yes"| W["Deterministic workflow"]
    A -->|"No, representative evidence"| B{"Does one bounded reasoner suffice?"}
    B -->|"Yes"| S["Single bounded agent"]
    B -->|"No, measured decomposition benefit"| M["Multi-agent with typed envelopes"]
    W --> H["Human approval before write"]
    S --> G["Policy, budgets, tools, audit outside model"]
    M --> G2["Same controls plus convergence and delegation bounds"]
    style A fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style W fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style M fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G2 fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Common Pitfalls

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Call a fixed pipeline an agent because it contains an LLM | Model use and model-owned control flow are different properties |
| Right | Name who chooses each next step | The workflow/agent boundary becomes reviewable |
| Wrong | Create one agent per department | Organization charts do not prove reasoning decomposition |
| Right | Require measured coordination benefit and typed authority envelopes | Multi-agent cost and trust boundaries must earn their keep |

**Quick Health Check:** reconsider one agent only when a valuable branch is genuinely unenumerable in representative cases. Reconsider multiple agents only after one bounded agent is justified and becomes a measured bottleneck.

**Your turn:** change the one variable in the next cell only when you can attach representative evidence. The check may move from workflow to one bounded-agent review, never directly to multi-agent.

In [ ]:
# ── Apply the Agency Test and Reopen It Carefully ────────────────────────
AGENCY_TEST = [
    {"use_case_id": "UC-RIV-001", "route": "authorize -> retrieve -> cite -> answer or abstain", "model_selects_next_tool": False},
    {"use_case_id": "UC-RIV-002", "route": "authorize selected title -> draft -> review", "model_selects_next_tool": False},
    {"use_case_id": "UC-RIV-003", "route": "validate transition -> propose -> confirm payload -> commit or stop", "model_selects_next_tool": False},
    {"use_case_id": "UC-RIV-004", "route": "authorize -> retrieve restriction -> counsel decides", "model_selects_next_tool": False},
]
fixture_use_case_ids = {item["use_case_id"] for item in case["use_cases"]}
assert {item["use_case_id"] for item in AGENCY_TEST} == fixture_use_case_ids
assert all(not item["model_selects_next_tool"] for item in AGENCY_TEST)
recommendation = "deterministic_workflow"
print(f"Prediction resolved: {recommendation} owns control flow.")
print("PASS: all four canonical routes are enumerable without model-selected tools.")

# CHANGE THIS only when a representative case proves the next action cannot be enumerated.
REPRESENTATIVE_UNENUMERABLE_BRANCH = False
if REPRESENTATIVE_UNENUMERABLE_BRANCH:
    next_review = "single bounded agent: define tools, budgets, termination, evaluation, and escalation"
else:
    next_review = "deterministic workflow: keep model calls inside fixed nodes"
assert "multi-agent" not in next_review
print(f"Exercise decision path: {next_review}")
print("Takeaway: one new uncertainty can reopen one adjacent decision; it cannot skip the evidence ladder.")

## 4 - Draw Boundaries Before Products

Riverside's required request context is `tenant_id`, `actor_id`, `role_ids`, `region_id`, `purpose`, `title_ids`, and `trace_id`. Trusted ingress validates it; retrieval and tools consume it; audit preserves a bounded record. Product names come later.

```mermaid
flowchart LR
    U["Editor"] --> I["Trusted ingress + identity context"]
    I --> P{"Deterministic policy"}
    P -->|"policy lookup"| R["Lifecycle + ACL pre-filtered retrieval"]
    R --> C["Cited answer model in approved region"]
    P -->|"continuation"| T["Explicit title context"]
    T --> G["Bounded drafting model in approved region"]
    P -->|"workflow proposal"| W["Deterministic transition builder"]
    W --> H["Human approves exact payload"]
    H --> X["PageTurn adapter: initially disabled"]
    C --> E["Editor decides use"]
    G --> E2["Editor accepts or rejects"]
    R --> L["Rights counsel interprets restrictions"]
    I --> A["Append-only bounded audit"]
    P --> A
    H --> A
    X --> A
    style U fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style T fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style W fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E2 fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style L fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### The Workflow Write Boundary

`UC-RIV-003` is a proposal path, not permission for autonomous writes. Approval binds to the exact title, prior state, next state, actor, and business idempotency key. A timeout is an ambiguous outcome: query PageTurn for committed state before retrying. Because `UNK-RIV-005` is open, the first release stops before the adapter call.

```mermaid
sequenceDiagram
    participant E as Editor
    participant W as Deterministic Workflow
    participant P as Policy
    participant A as Approval Store
    participant T as PageTurn Adapter
    participant S as PageTurn
    E->>W: request transition proposal
    W->>P: validate actor, title, prior and next state
    P-->>W: require exact-payload approval
    W->>E: show exact payload and impact
    E->>A: approve payload hash
    A-->>W: approval bound to payload and state version
    W->>T: submit with one business idempotency key
    Note over W,T: Disabled while UNK-RIV-005 remains open
    T->>S: conditional transition
    S--xT: response may be lost after commit
    T->>S: reconcile by business key before retry
```

### Common Pitfalls

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Put tenant or title filters in the prompt | A persuaded model can omit or alter them |
| Right | Filter before retrieval and inference | Unauthorized content never enters model context |
| Wrong | Treat approval as `yes, update it` | Arguments can change before execution |
| Right | Hash-bind approval to exact current arguments | A mismatch forces re-approval |
| Wrong | Retry PageTurn after timeout with a new key | The first call may already have committed |
| Right | Reuse one business key and reconcile first | Ambiguous outcomes become state checks, not guesses |

**Quick Health Check:** remove any required context field, mutate one approved argument, or leave the idempotency unknown open. Each case must block the write.

In [ ]:
# ── Build ARC-02 and Prove Fail-Closed Approval ──────────────────────────
REQUIRED_CONTEXT = set(case["identity_and_data_constraints"]["required_request_context"])
BOUNDARIES = [
    {"id": "BND-RIV-IDENTITY", "type": "identity", "control": "validate required context; fail closed", "human_owner": "PER-RIV-004"},
    {"id": "BND-RIV-DATA", "type": "data", "control": "filter lifecycle, tenant, region, purpose, role, and title before ranking", "human_owner": "source owner"},
    {"id": "BND-RIV-MODEL", "type": "model", "control": "approved regional route; bounded input/output; no authority", "human_owner": "PER-RIV-004"},
    {"id": "BND-RIV-POLICY", "type": "policy", "control": "deterministic allow, deny, or require approval", "human_owner": "policy owner"},
    {"id": "BND-RIV-HUMAN", "type": "human", "control": "bind decision to exact current payload", "human_owner": "authorized reviewer"},
    {"id": "BND-RIV-TOOL", "type": "tool", "control": "scoped identity, idempotency, reconciliation, audit, recovery", "human_owner": "PER-RIV-005"},
    {"id": "BND-RIV-STATE", "type": "state", "control": "versioned transition and append-only decision evidence", "human_owner": "PER-RIV-005"},
    {"id": "BND-RIV-EXTERNAL", "type": "external_validation", "control": "block claims until target-environment evidence exists", "human_owner": "PER-FDE-001"},
]
assert REQUIRED_CONTEXT == {"tenant_id", "actor_id", "role_ids", "region_id", "purpose", "title_ids", "trace_id"}
assert {item["type"] for item in BOUNDARIES} == {"identity", "data", "model", "policy", "human", "tool", "state", "external_validation"}
assert all(item["control"] and item["human_owner"] for item in BOUNDARIES)

def canonical_hash(payload):
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

def context_allowed(request_context):
    return REQUIRED_CONTEXT.issubset(request_context) and all(request_context[field] for field in REQUIRED_CONTEXT)

valid_context = {"tenant_id": "TEN-RIV-EU", "actor_id": "API-USER-EDITOR-017", "role_ids": ["ROLE-SENIOR-EDITOR"], "region_id": "REG-UKS", "purpose": "workflow_assistance", "title_ids": ["TITLE-ARIA"], "trace_id": "trace-synthetic-001"}
payload = {"task_id": "API-TASK-9003", "title_id": "TITLE-ARIA", "from_state": "copy_edit", "to_state": "senior_review", "actor_id": "API-USER-EDITOR-017", "business_key": "transition-9003-copy-edit-senior-review"}
approval_hash = canonical_hash(payload)
mutated_payload = {**payload, "to_state": "approved_for_publication"}
assert context_allowed(valid_context)
assert not context_allowed({key: value for key, value in valid_context.items() if key != "purpose"})
assert approval_hash != canonical_hash(mutated_payload)
assert next(item for item in case["unknowns"] if item["unknown_id"] == "UNK-RIV-005")["status"] == "open"
pprint(BOUNDARIES)
print("PASS: all boundary types are owned; missing purpose and payload mutation block the PageTurn write.")
print("Takeaway: policy and approval operate on exact state; model confidence is irrelevant.")

## 5 - The Smallest Riverside Architecture

The selected design is a deterministic application with independently gated paths, not a general agent runtime. Policy lookup and rights lookup share authorized retrieval. Policy answers may add cited generation; rights interpretation stays with counsel. Continuation drafting uses one explicitly selected title and a region-approved bounded model route. Workflow writes remain disabled until PageTurn recovery evidence exists.

```mermaid
flowchart TD
    P0["Phase 0: lifecycle cleanup + manual/search fallback"] --> P1["Phase 1: read and draft only"]
    P1 --> L["Policy: authorized search + optional cited synthesis"]
    P1 --> C["Continuation: selected title + bounded private prompt"]
    P1 --> R["Rights: authorized lookup + counsel decision"]
    L --> G1["Retrieval, citation, isolation, latency gates"]
    C --> G2["Instruction, style, safety, isolation, latency gates"]
    R --> G3["Negative access + source freshness gates"]
    G1 --> P2["Phase 2 candidate: confirmed workflow proposal"]
    G2 --> P2
    G3 --> P2
    P2 --> B{"UNK-RIV-005 closed and duplicate-commit test passes?"}
    B -->|"No"| D["Keep writes disabled"]
    B -->|"Yes"| W["Canary exact-payload workflow writes"]
    style P0 fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P1 fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style L fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G1 fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G2 fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G3 fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P2 fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style W fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Anti-AI Challenge

1. Can lifecycle cleanup plus search meet `UC-RIV-001` without generation? If evaluation says yes, remove cited generation.
2. Is continuation time saving large enough to justify any approved model route? If not, keep drafting manual.
3. Does a model improve a decision, or merely smooth prose? Smoother prose cannot relax evidence, access, or review.
4. During a regional model outage, UK/EU manuscript requests fail closed while authorized policy search may remain.

### Common Pitfalls

| | Pattern | Why it matters |
|---|---|---|
| Wrong | One large platform for all future use cases | Components without current criteria add cost and risk |
| Right | Independent feature flags and release IDs per path | Disable generation or writes while keeping safe search |
| Wrong | Cross-region failover because availability matters | UK/EU confidential processing has no approved alternate region |
| Right | Fail closed or degrade to policy-only mode | Availability cannot silently override residency |

**Quick Health Check:** every component maps to a use case, test, rollout gate, operating signal, and handoff owner. Every generative path can be disabled without disabling manual work or authorized source search.

In [ ]:
# ── Prove Component Traceability and AI-Off Degradation ─────────────────
SELECTED_COMPONENTS = {
    "deterministic_control_shell": {"UC-RIV-001", "UC-RIV-002", "UC-RIV-003", "UC-RIV-004"},
    "authorized_search": {"UC-RIV-001", "UC-RIV-004"},
    "cited_rag_answer": {"UC-RIV-001"},
    "bounded_private_prompt": {"UC-RIV-002"},
    "human_approval_workflow": {"UC-RIV-003"},
    "append_only_decision_evidence": {"UC-RIV-001", "UC-RIV-002", "UC-RIV-003", "UC-RIV-004"},
}
REJECTED_COMPONENTS = {"fine_tuning", "single_agent", "multi_agent"}
all_mapped_use_cases = set().union(*SELECTED_COMPONENTS.values())
assert all_mapped_use_cases == set(USE_CASE_REQUIREMENTS)
assert REJECTED_COMPONENTS.isdisjoint(SELECTED_COMPONENTS)

AI_OFF_CAPABILITIES = {"manual_process", "source_lifecycle", "authorized_current_sources", "authorized_rights_lookup"}
AI_ONLY_CAPABILITIES = {"cited_answer", "bounded_generation"}
open_unknowns = {item["unknown_id"]: item["status"] for item in case["unknowns"]}
assert {"manual_process", "authorized_current_sources"}.issubset(AI_OFF_CAPABILITIES)
assert AI_OFF_CAPABILITIES.isdisjoint(AI_ONLY_CAPABILITIES)
assert open_unknowns["UNK-RIV-005"] == "open"
assert open_unknowns["UNK-RIV-008"] == "external_validation_required"
for component, mapped_use_cases in SELECTED_COMPONENTS.items():
    print(f"{component:<35} -> {', '.join(sorted(mapped_use_cases))}")
print("PASS: every selected component maps to a use case; AI-off mode preserves manual work and authorized lookup.")
print("PASS: workflow writes and production model routing remain blocked by named evidence gaps.")

## 6 - ADR-001 and the Customer Explanation

```mermaid
flowchart LR
    C["Context: four bounded use cases"] --> D["Decision: deterministic shell + smallest capability per use case"]
    D --> S["Selected: process, search/RAG, bounded prompt, workflow design"]
    D --> R["Rejected now: fine-tuning, one agent, multiple agents"]
    S --> G["Gate: data, identity, evaluation, rollout, recovery evidence"]
    R --> T["Revisit only on named measured trigger"]
    style C fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style T fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### ADR-001: Select a Phased Deterministic Application

- **Status:** Proposed
- **Decision owners:** Maya Chen for launch scope, Elena Marlow for editorial workflow, Aisha Rahman for security and regional controls, Theo Grant for integration operations
- **Fixture scope:** `RIV-FDE-1.0.0`
- **Revalidate on:** changed use cases, role model, regional policy, source contract, PageTurn behavior, evaluation evidence, or support ownership

**Context.** Riverside needs faster current-policy answers and bounded manuscript drafting. Canonical routes and authority boundaries are enumerable. Current evidence does not justify model-selected tools or distributed reasoning. Public generative AI is prohibited for unpublished manuscript content, UK/EU and US routes differ, and PageTurn idempotency remains unknown.

**Decision.** Build a deterministic application shell. Phase 0 repairs source lifecycle and preserves manual/search fallback. Phase 1 offers read-only authorized policy/RAG answers, bounded private continuation drafting, and read-only rights lookup. Design but do not enable exact-payload PageTurn writes until `UNK-RIV-005` is resolved and duplicate-commit failure injection passes. Defer fine-tuning, a single agent, and multiple agents.

**Positive consequences.** Smaller test surface; explicit authority; independent feature disablement; current evidence remains outside weights; manual and search-only modes survive model failure.

**Negative consequences.** The product will not handle arbitrary automation; editors still confirm writes and accept drafts; paths need separate evaluation; regional inference may reduce availability.

**Revisit triggers.** Fine-tuning requires a measured stable behavior gap. One agent requires a measured valuable unenumerable branch. Multiple agents require a justified single agent plus a measured context, ownership, or throughput bottleneck and coordination gain.

### ARC-03: Customer-Readable Explanation

Riverside's editors currently spend a median of 18 minutes finding policy guidance and 42 minutes preparing a bounded continuation in the supplied samples. We recommend improving those tasks separately instead of introducing a general autonomous assistant.

For policy questions, the service checks the editor's imprint, role, region, purpose, and title access. It searches only current approved sources and shows the passages used. A bounded language model may turn those passages into a concise cited answer, but the editor decides whether to use it. If generation is unavailable or not approved, authorized search remains available.

For manuscript continuations, the editor selects the title and supplies a bounded instruction. Only content for that assigned title can enter an approved model route in the title's permitted region. The result is a draft: it is not saved, published, or distributed until an editor chooses to do so through existing controls.

Rights restrictions remain read-only evidence for Rights counsel. The service will not grant rights, change contracts, alter payments, publish text, or infer a missing right. A workflow change may be proposed, but no write is enabled until Riverside proves how PageTurn deduplicates and reconciles a request whose response is lost. A human will later review the exact title, prior state, next state, actor, and operation key before any submission.

This recommendation does not prove model quality, regional capacity, latency, availability, price, deletion behavior, security approval, compliance, or support readiness. Those items remain validation work for Riverside's authorized owners.

In [ ]:
# ── Check ADR and Customer Explanation Discipline ────────────────────────
ADR_001 = {
    "status": "proposed",
    "fixture_version": "RIV-FDE-1.0.0",
    "selected": ["process_repair", "deterministic_control_shell", "authorized_search", "cited_rag_answer", "bounded_private_prompt", "workflow_design_writes_disabled"],
    "rejected": {
        "fine_tuning": "stable behavior gap measured",
        "single_agent": "valuable unenumerable branch measured",
        "multi_agent": "single-agent bottleneck and coordination gain measured",
    },
    "blocked_by": ["UNK-RIV-005", "UNK-RIV-008"],
    "ai_off_fallback": "manual workflow plus authorized current-source search",
}
CUSTOMER_EXPLANATION_CHECKS = {
    "names_two_baselines": True,
    "names_human_decisions": True,
    "names_ai_off_search": True,
    "blocks_rights_and_publication_authority": True,
    "keeps_pageturn_write_disabled": True,
    "names_external_validation": True,
    "claims_customer_approval": False,
    "claims_production_readiness": False,
}
assert ADR_001["status"] == "proposed"
assert set(ADR_001["rejected"]) == REJECTED_COMPONENTS
assert all(trigger.endswith("measured") for trigger in ADR_001["rejected"].values())
assert set(ADR_001["blocked_by"]) == {"UNK-RIV-005", "UNK-RIV-008"}
assert all(value for key, value in CUSTOMER_EXPLANATION_CHECKS.items() if not key.startswith("claims_"))
assert not any(value for key, value in CUSTOMER_EXPLANATION_CHECKS.items() if key.startswith("claims_"))
print("PASS: ADR-001 is proposed, bounded, reversible, and explicit about rejected complexity.")
print("PASS: ARC-03 states controls, fallback, limitations, and next evidence without inventing approval.")

## 7 - Proof Exercises and Architecture Health

The architecture is a hypothesis. These exercises attack boundaries most likely to be weakened by schedule pressure or an impressive demo.

```mermaid
flowchart LR
    A["Proposed architecture"] --> P1["Remove one context field"]
    A --> P2["Swap current policy for superseded source"]
    A --> P3["Mutate approved workflow payload"]
    A --> P4["Disable all model routes"]
    P1 --> B["Must deny"]
    P2 --> B
    P3 --> B
    P4 --> D["Must degrade to manual/search mode"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P1 fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P2 fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P3 fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P4 fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Your turn:** change one attack in the next cell. A safe response either remains allowed, fails closed, or opens a named review. It never silently grants model authority or skips from workflow directly to multiple agents.

### Final Common Pitfalls

| Pitfall | Safer practice |
|---|---|
| Architecture by product catalog | Architecture by use case, invariant, boundary, and evidence |
| Weighted score overrides security failure | Hard constraints veto the candidate before scoring |
| One average quality metric | Separate retrieval, citation, generation, policy, latency, cost, and review slices |
| Fine-tuned fluency treated as knowledge | Current facts stay in versioned authorized retrieval |
| Prompt instruction treated as access control | Identity and policy checks execute outside the model |
| Deployment rollback treated as action undo | Reconcile and compensate committed effects separately |
| Agent added for flexibility | Add one only for measured unenumerable runtime branching |
| Multi-agent added for organizational symmetry | Add many only for measured coordination benefit |
| Production claim from local fixture | Label local result and assign target-environment validation |
| AI outage means service outage | Preserve manual workflow and authorized search-only mode |

### Quick Health Check

1. Every selected component maps to `UC-RIV-001` through `UC-RIV-004`.
2. Every rejected option has a measured revisit trigger.
3. All seven request-context fields fail closed.
4. Every boundary has a deterministic control and human owner.
5. Model-backed options own no authority capability.
6. `UNK-RIV-005` blocks PageTurn writes.
7. `UNK-RIV-008` blocks production routing, quota, price, and regional claims.
8. AI-off mode preserves manual work and authorized lookup.
9. The ADR remains proposed and invents no customer validation.
10. The customer explanation states limitations and next evidence.

In [ ]:
# ── Attack a Boundary and Run the Full Health Check ─────────────────────
# CHANGE THIS to: missing_context, stale_source, changed_payload, model_outage, or unenumerable_branch.
ATTACK = "missing_context"
EXPECTED_RESPONSE = {
    "missing_context": "deny",
    "stale_source": "exclude_before_ranking",
    "changed_payload": "require_reapproval",
    "model_outage": "manual_and_search_degraded_mode",
    "unenumerable_branch": "open_single_agent_review",
}
if ATTACK not in EXPECTED_RESPONSE:
    raise ValueError(f"Unknown exercise attack: {ATTACK}")
response = EXPECTED_RESPONSE[ATTACK]
assert response != "multi_agent"
print(f"Attack: {ATTACK}; expected response: {response}")

HEALTH_CHECKS = {
    "all_use_cases_mapped": all_mapped_use_cases == set(USE_CASE_REQUIREMENTS),
    "measured_revisit_triggers": all(trigger.endswith("measured") for trigger in ADR_001["rejected"].values()),
    "required_context_complete": REQUIRED_CONTEXT == {"tenant_id", "actor_id", "role_ids", "region_id", "purpose", "title_ids", "trace_id"},
    "boundary_owners_present": all(item["control"] and item["human_owner"] for item in BOUNDARIES),
    "models_do_not_own_authority": all(not (OPTIONS[name] & AUTHORITY_CAPABILITIES) for name in MODEL_OPTIONS),
    "pageturn_write_blocked": open_unknowns["UNK-RIV-005"] == "open",
    "production_routing_external": open_unknowns["UNK-RIV-008"] == "external_validation_required",
    "ai_off_path_exists": {"manual_process", "authorized_current_sources"}.issubset(AI_OFF_CAPABILITIES),
    "adr_is_proposed": ADR_001["status"] == "proposed",
    "customer_explanation_bounded": not CUSTOMER_EXPLANATION_CHECKS["claims_customer_approval"] and not CUSTOMER_EXPLANATION_CHECKS["claims_production_readiness"],
}
for check_name, passed in HEALTH_CHECKS.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check_name}")
assert all(HEALTH_CHECKS.values())
print("Takeaway: local structure is internally consistent; target-environment behavior remains unvalidated.")

## Roadmap Checkpoint

```mermaid
flowchart LR
    A["Ambiguous AI brief"] --> B["Nine options compared"]
    B --> C["Smallest phased composition selected"]
    C --> D["Boundaries and authority explicit"]
    D --> E["Next: prove data readiness and isolation"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

| Constraint | Before | Architecture-stage result | Evidence status |
|---|---|---|---|
| Solution choice | Implied broad AI automation | Phased deterministic composition | Proposed teaching decision |
| Policy lookup | 18-minute median; 61% current-first | Authorized search plus optional cited synthesis | Local fixture execution verified; customer evaluation still required |
| Continuation | 42-minute median | Explicit-title bounded private prompt path | Requires approved route and evaluation |
| Workflow write | Desired but recovery unknown | Designed, disabled until `UNK-RIV-005` closes | Blocked |
| Agentic control | Assumed by solution language | Rejected: canonical routes are enumerable | Revisit on measured unenumerable branch |
| Multi-agent | No requirement | Rejected: no coordination benefit demonstrated | Revisit after one agent is justified and measured |

### Coverage Ledger

| Tier | Techniques | Reason |
|---|---|---|
| Built and checked | No-AI review, deterministic controls, search, RAG, prompt call, fine-tuning boundary, workflow/agent test, multi-agent test, option matrix, boundary register, ADR, AI-off mode | These mechanisms determine the Riverside decision |
| Explained and illustrated | Hybrid retrieval, groundedness, policy-as-code, approval hashing, reconciliation, feature flags, regional degraded mode | Their architecture role is needed here; implementation belongs elsewhere |
| Named with a reason | Provider selection, embedding choice, fine-tuning recipe, index tuning, cloud topology, multi-agent protocol | Each requires data, evaluation, quota, or runtime evidence not available here |

If a technique named above is missing from this ledger, that is the coverage bug this table exists to catch.

### Key Takeaways

- The smallest valid architecture is often a composition, not one fashionable label.
- Fine-tune stable behavior; retrieve current authorized evidence; keep authority deterministic and human-owned.
- A model inside a fixed node does not make the workflow an agent.
- Add one agent only for proven runtime branch uncertainty; add many only for proven coordination benefit.
- Manual and search-only paths are architecture features, not signs of failure.
- Unknowns block exposure; they do not become convenient assumptions.

> **Forward:** data onboarding must prove lifecycle, ACL, deletion, and title mapping for selected retrieval paths. Identity and isolation must prove required context survives every boundary. Until those gates pass, `ADR-001` remains proposed.